In [0]:
SELECT
    COUNT(*) AS row_count,
    SUM(size(orders)) AS orders_count
FROM business_lakehouse.bronze.orders VERSION AS OF 0;

SELECT
    COUNT(*) AS row_count,
    SUM(size(orders)) AS orders_count
FROM business_lakehouse.bronze.orders VERSION AS OF 1;

SHOW TABLES IN business_lakehouse.bronze
LIKE 'products_incremental';

SELECT COUNT(*)
FROM business_lakehouse.bronze.products_incremental;

SELECT
    COUNT(*) AS row_count,
    SUM(size(products)) AS products_count
FROM business_lakehouse.bronze.products_incremental;

SELECT
    sourceSystem,
    size(products) AS products_count,
    _source_file
FROM business_lakehouse.bronze.products_incremental;

SELECT
    COUNT(*) AS row_count,
    SUM(size(products)) AS products_count
FROM business_lakehouse.bronze.products_rescue_test;

SELECT
    _source_file,
    _rescued_data
FROM business_lakehouse.bronze.products_rescue_test
WHERE _rescued_data IS NOT NULL;

CREATE TABLE IF NOT EXISTS business_lakehouse.bronze.products_copy_test (
    products VARIANT,
    limit BIGINT,
    skip BIGINT,
    total BIGINT,
    sourceSystem STRING
)
USING DELTA;

COPY INTO business_lakehouse.bronze.products_copy_test
FROM (
    SELECT
        parse_json(to_json(products)) AS products,
        limit,
        skip,
        total,
        sourceSystem
    FROM '/Volumes/business_lakehouse/landing/raw_volume/ecommerce/products/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS ('multiLine' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

SELECT
    COUNT(*) AS row_count,
    COUNT_IF(products IS NOT NULL) AS non_null_products
FROM business_lakehouse.bronze.products_copy_test;

SELECT
    COUNT(*) AS products_count,
    COUNT(DISTINCT id) AS unique_products
FROM business_lakehouse.bronze.products_flattened;

SELECT *
FROM business_lakehouse.bronze.products_flattened
LIMIT 10;

SELECT
    id,
    title,
    availabilityStatus,
    stock
FROM business_lakehouse.bronze.products_quality_test
WHERE availabilityStatus IS NULL
   OR availabilityStatus NOT IN (
       'In Stock',
       'Low Stock',
       'Out of Stock'
   );

   SELECT
    availabilityStatus,
    COUNT(*) AS products_count
FROM business_lakehouse.bronze.products_quality_test
GROUP BY availabilityStatus
ORDER BY products_count DESC;

SELECT
    id,
    title,
    availabilityStatus
FROM business_lakehouse.bronze.products_quality_test
WHERE availabilityStatus IS NULL;

SELECT
    COUNT(*) AS source_rows,
    SUM(SIZE(products)) AS total_products
FROM business_lakehouse.bronze.products_streaming;

SELECT
    COUNT(*) AS source_rows,
    SUM(SIZE(users)) AS total_users
FROM business_lakehouse.bronze.users_streaming;

SELECT
    COUNT(*) AS source_rows,
    SUM(SIZE(orders)) AS total_orders
FROM business_lakehouse.bronze.orders_streaming;

DESCRIBE TABLE business_lakehouse.bronze.exchange_rates_streaming;

SELECT *
FROM business_lakehouse.bronze.exchange_rates_streaming
LIMIT 3;

DESCRIBE DETAIL business_lakehouse.bronze.products_streaming;
DESCRIBE DETAIL business_lakehouse.bronze.users_streaming;
DESCRIBE DETAIL business_lakehouse.bronze.orders_streaming;
DESCRIBE DETAIL business_lakehouse.bronze.exchange_rates_streaming;

SELECT
    COUNT(*) AS source_rows,
    SUM(SIZE(Cube)) AS total_days,
    SUM(
        AGGREGATE(
            Cube,
            CAST(0 AS BIGINT),
            (acc, day) -> acc + SIZE(day.Cube)
        )
    ) AS total_exchange_rates,
    COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
FROM business_lakehouse.bronze.exchange_rates_streaming;

SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_time,
    COUNT_IF(_source_file IS NULL) AS missing_source_file,
    MIN(_ingested_at) AS first_ingestion,
    MAX(_ingested_at) AS last_ingestion
FROM business_lakehouse.bronze.products_streaming;

